# CARGA DEL ARCHIVO

In [1]:
# ===============================
# 1. IMPORTAR LIBRERÍAS
# ===============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
df_redshift = pd.read_csv("combined_data_with_redshift_V8.csv")
display(df_redshift.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           251 non-null    object 
 1   Redshift_crosscheck  251 non-null    float64
 2   T90                  251 non-null    float64
 3   log10Fluence         251 non-null    float64
 4   log10PeakFlux        246 non-null    float64
 5   PhotonIndex          251 non-null    float64
 6   log10NH              251 non-null    float64
 7   Gamma                251 non-null    float64
 8   log10Fa              251 non-null    float64
 9   log10Ta              251 non-null    float64
 10  Alpha                251 non-null    float64
 11  Beta                 251 non-null    float64
 12  log10TaErr           251 non-null    float64
 13  BetaErr              251 non-null    float64
 14  AlphaErr             251 non-null    float64
 15  T90Err               250 non-null    flo

None

# 1. Eliminación de SGRB

Eliminamos todos aquellos GRBs cuyo valor de la variable $T_{90} < 2$

In [3]:
df_redshift = df_redshift[df_redshift["T90"]>=2]

In [4]:
for i in df_redshift["T90"]:
    if i < 2:
        print("hay un SGRB")

# 2. Features seleccionadas

Nos quedamos con las mismas features que el modelo de Narendra

In [5]:
df_redshift = df_redshift[["Redshift_crosscheck","log10NH","log10PeakFlux", "PhotonIndex", "log10Ta",
    "log10Fa", "Gamma", "Alpha"]]
print(df_redshift.info())


<class 'pandas.core.frame.DataFrame'>
Index: 238 entries, 0 to 250
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Redshift_crosscheck  238 non-null    float64
 1   log10NH              238 non-null    float64
 2   log10PeakFlux        233 non-null    float64
 3   PhotonIndex          238 non-null    float64
 4   log10Ta              238 non-null    float64
 5   log10Fa              238 non-null    float64
 6   Gamma                238 non-null    float64
 7   Alpha                238 non-null    float64
dtypes: float64(8)
memory usage: 16.7 KB
None


In [6]:
# Porcentaje de valores NO nulos (completitud)
completitud = (df_redshift.notnull().mean() * 100).round(2)
completitud = completitud.sort_values(ascending=False)

print("Completitud (% de filas con dato):")
print(completitud)

Completitud (% de filas con dato):
Redshift_crosscheck    100.0
log10NH                100.0
PhotonIndex            100.0
log10Ta                100.0
Gamma                  100.0
log10Fa                100.0
Alpha                  100.0
log10PeakFlux           97.9
dtype: float64


# TRATAMIENTO DE LAS CARACTERÍSTICAS


Marcar como nulos:
• log10NH < 20.
• α, β o γ > 3.
• PhotonIndex < 0

In [7]:
# Suponiendo que tienes una lista con los nombres originales
nombres_columnas = ['Redshift_crosscheck', 'log10NH', 'log10PeakFlux', 'PhotonIndex', 'log10Ta', 'log10Fa', 'Gamma', 'Alpha']
df_redshift = pd.DataFrame(df_redshift, columns=nombres_columnas)
print(type(df_redshift))
display(nombres_columnas)

<class 'pandas.core.frame.DataFrame'>


['Redshift_crosscheck',
 'log10NH',
 'log10PeakFlux',
 'PhotonIndex',
 'log10Ta',
 'log10Fa',
 'Gamma',
 'Alpha']

In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# df original (ya con todas las columnas numéricas limpias de puntos, etc.)
# 1. Marcar outliers como NaN
df_redshift.loc[df_redshift['log10NH'] < 20, 'log10NH'] = np.nan
df_redshift.loc[df_redshift['Alpha'] > 3, 'Alpha'] = np.nan
df_redshift.loc[df_redshift['Gamma'] > 3, 'Gamma'] = np.nan
df_redshift.loc[df_redshift['PhotonIndex'] < 0, 'PhotonIndex'] = np.nan




In [9]:
# Guardar nombres de columnas e índice
columnas = df_redshift.columns
indice = df_redshift.index

# Aplicar imputación
imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42)
datos_imputados = imputer.fit_transform(df_redshift)

# Reconstruir DataFrame
df_redshift = pd.DataFrame(datos_imputados, columns=columnas, index=indice)
print(type(df_redshift))
display(df_redshift.info())

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
Index: 238 entries, 0 to 250
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Redshift_crosscheck  238 non-null    float64
 1   log10NH              238 non-null    float64
 2   log10PeakFlux        238 non-null    float64
 3   PhotonIndex          238 non-null    float64
 4   log10Ta              238 non-null    float64
 5   log10Fa              238 non-null    float64
 6   Gamma                238 non-null    float64
 7   Alpha                238 non-null    float64
dtypes: float64(8)
memory usage: 16.7 KB


None

In [10]:
# 3. Imputar con MICE (20 iteraciones)
imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    imputation_order='ascending',
    skip_complete=True
)

# Guardar nombres de columnas e índice ANTES de transformar
columnas = df_redshift.columns
indice = df_redshift.index

# Aplicar imputación (devuelve numpy array)
datos_imputados = imputer.fit_transform(df_redshift)

# Reconstruir DataFrame
df_redshift = pd.DataFrame(datos_imputados, columns=columnas, index=indice)

# 4. Verificar resultados
print("Nulos después de imputación:\n", df_redshift.isna().sum())
print("\nEstadísticas descriptivas:")
print(df_redshift.describe())

Nulos después de imputación:
 Redshift_crosscheck    0
log10NH                0
log10PeakFlux          0
PhotonIndex            0
log10Ta                0
log10Fa                0
Gamma                  0
Alpha                  0
dtype: int64

Estadísticas descriptivas:
       Redshift_crosscheck     log10NH  log10PeakFlux  PhotonIndex  \
count           238.000000  238.000000     238.000000   238.000000   
mean              2.139020   21.540213       0.444832     1.602117   
std               1.370692    0.506056       0.546143     0.401720   
min               0.033100   20.028803      -0.602060     0.490000   
25%               1.114650   21.167576       0.079181     1.340000   
50%               1.921450   21.569987       0.350416     1.580000   
75%               3.012750   21.913342       0.796016     1.850000   
max               8.260000   22.863321       2.120574     3.080000   

          log10Ta     log10Fa       Gamma       Alpha  
count  238.000000  238.000000  238.000000 

In [11]:
print(df_redshift.head())

   Redshift_crosscheck    log10NH  log10PeakFlux  PhotonIndex   log10Ta  \
0                1.949  21.916838       0.285557         2.11  4.863560   
1                1.440  20.926617       0.499687         1.90  4.132085   
2                3.240  20.937354       0.181844         2.02  3.739970   
3                2.900  22.125702       1.029384         1.40  3.124350   
4                4.270  22.230630       0.267172         1.41  4.270584   

     log10Fa    Gamma     Alpha  
0 -11.375718  1.99544  1.108180  
1 -11.154902  1.98053  1.874408  
2 -10.705534  2.01045  0.863159  
3  -9.653647  1.78711  0.984108  
4 -11.048177  2.01835  1.462095  


## Convertir Redshift


In [12]:
# Crear la nueva columna log10(Redshift_crosscheck + 1)
df_redshift['log Redshift'] = np.log10(df_redshift['Redshift_crosscheck'] + 1)

# Verifica los primeros valores
print(df_redshift[['Redshift_crosscheck', 'log Redshift']].head())

df_redshift = df_redshift.drop(columns=["Redshift_crosscheck"])

print(df_redshift.info())

   Redshift_crosscheck  log Redshift
0                1.949      0.469675
1                1.440      0.387390
2                3.240      0.627366
3                2.900      0.591065
4                4.270      0.721811
<class 'pandas.core.frame.DataFrame'>
Index: 238 entries, 0 to 250
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   log10NH        238 non-null    float64
 1   log10PeakFlux  238 non-null    float64
 2   PhotonIndex    238 non-null    float64
 3   log10Ta        238 non-null    float64
 4   log10Fa        238 non-null    float64
 5   Gamma          238 non-null    float64
 6   Alpha          238 non-null    float64
 7   log Redshift   238 non-null    float64
dtypes: float64(8)
memory usage: 16.7 KB
None


# 5. M - Estimator

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 1 — Variables (ya las tienes declaradas)
# ══════════════════════════════════════════════════════════════════════════════
log_NH       = df_redshift["log10NH"]
log_PeakFlux = df_redshift["log10PeakFlux"]
log_Ta       = df_redshift["log10Ta"]
log_Fa       = df_redshift["log10Fa"]
photon_index = df_redshift["PhotonIndex"]
gamma        = df_redshift["Gamma"]
alpha        = df_redshift["Alpha"]
y            = df_redshift["log Redshift"]

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 2 — Construcción de la matriz X (Ecuación 1 de Narendra)
# ══════════════════════════════════════════════════════════════════════════════
inner        = (log_Fa**2) + log_Fa + log_PeakFlux
complex_term = inner**2

X = pd.DataFrame({
    'complex_term'    : complex_term,
    'log_NH'          : log_NH,
    'PhotonIndex'     : photon_index,
    'log_Ta'          : log_Ta,
    'gamma'           : gamma,
    'alpha'           : alpha,
    'log_NH_sq'       : log_NH**2,
    'log_PeakFlux_sq' : log_PeakFlux**2,
    'PhotonIndex_sq'  : photon_index**2,
    'log_Ta_sq'       : log_Ta**2,
    'gamma_sq'        : gamma**2,
    'alpha_sq'        : alpha**2,
})
X = sm.add_constant(X)

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 3 — Estandarización (estabiliza la matriz numéricamente)
# ══════════════════════════════════════════════════════════════════════════════
X_sin_intercepto = X.drop(columns=['const'])
scaler           = StandardScaler()
X_scaled_values  = scaler.fit_transform(X_sin_intercepto)

X_scaled = pd.DataFrame(
    X_scaled_values,
    columns=X_sin_intercepto.columns,
    index=X_sin_intercepto.index
)
X_scaled = sm.add_constant(X_scaled)

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 4 — Búsqueda fina del t
# ══════════════════════════════════════════════════════════════════════════════
print("=== BÚSQUEDA FINA DEL PARÁMETRO t ===")
print(f"{'t':>6} | {'Eliminados':>10} | {'% eliminado':>12} | {'GRBs finales':>12}")
print("-" * 50)

mejor_t    = None
mejor_diff = np.inf

for t_val in np.arange(1.10, 1.21, 0.01):
    rlm_temp    = sm.RLM(y, X_scaled, M=sm.robust.norms.HuberT(t=t_val))
    result_temp = rlm_temp.fit()
    w_temp      = result_temp.weights
    eliminados  = (w_temp < 0.65).sum()
    pct         = eliminados / len(df_redshift) * 100
    print(f"{t_val:>6.3f} | {eliminados:>10} | {pct:>11.1f}% | {len(df_redshift)-eliminados:>12}")

    diff = abs((len(df_redshift) - eliminados) - 226)
    if diff < mejor_diff:
        mejor_diff = diff
        mejor_t    = t_val

print(f"\n→ Mejor t encontrado: {mejor_t:.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# BLOQUE 5 — Modelo final y corte de Narendra
# ══════════════════════════════════════════════════════════════════════════════
rlm_final     = sm.RLM(y, X_scaled, M=sm.robust.norms.HuberT(t=mejor_t))
result_final  = rlm_final.fit()
weights_final = result_final.weights

mask_final  = weights_final >= 0.65
df_filtered = df_redshift[mask_final].copy()

print(f"\n=== RESULTADO FINAL ===")
print(f"GRBs antes del corte : {len(df_redshift)}")
print(f"GRBs eliminados      : {(~mask_final).sum()}  ({(~mask_final).sum()/len(df_redshift)*100:.1f}%)")
print(f"GRBs tras el corte   : {len(df_filtered)}")
print(f"Objetivo del paper   : ~226")

=== BÚSQUEDA FINA DEL PARÁMETRO t ===
     t | Eliminados |  % eliminado | GRBs finales
--------------------------------------------------
 1.100 |         19 |         8.0% |          219
 1.110 |         19 |         8.0% |          219
 1.120 |         19 |         8.0% |          219
 1.130 |         18 |         7.6% |          220


 1.140 |         18 |         7.6% |          220
 1.150 |         16 |         6.7% |          222
 1.160 |         15 |         6.3% |          223
 1.170 |         13 |         5.5% |          225
 1.180 |          9 |         3.8% |          229
 1.190 |          9 |         3.8% |          229
 1.200 |          8 |         3.4% |          230

→ Mejor t encontrado: 1.170

=== RESULTADO FINAL ===
GRBs antes del corte : 238
GRBs eliminados      : 13  (5.5%)
GRBs tras el corte   : 225
Objetivo del paper   : ~226


ll


In [15]:
# 5. Guardar a CSV
df_filtered.to_csv('LGRBs limpio def.csv', index=False)
print("\nArchivo 'LGRBs limpio' guardado correctamente.")



Archivo 'LGRBs limpio' guardado correctamente.
